In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import mplfinance as mpf
import math
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking


In [2]:
zip_file_path = 'datos_competicion.zip'
extraction_path = 'data'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"'{zip_file_path}' unzipped to '{extraction_path}'")

'datos_competicion.zip' unzipped to 'data'


In [3]:
trades = pd.read_csv('data/trades_benchmark.csv')
# Convert to datetime (keeps timezone correctly)
trades["timestamp_open"] = pd.to_datetime(trades["dateOpen"], format="mixed")
# Remove timezone
trades["timestamp_open"] = trades["timestamp_open"].dt.tz_convert(None)
# Drop seconds and microseconds
trades["timestamp_open"] = trades["timestamp_open"].dt.floor("min")


trades["timestamp_close"] = pd.to_datetime(trades["dateClose"], format="mixed")
# Remove timezone
trades["timestamp_close"] = trades["timestamp_close"].dt.tz_convert(None)
# Drop seconds and microseconds
trades["timestamp_close"] = trades["timestamp_close"].dt.floor("min")

In [8]:
algoritmos_path = r'data/algoritmos'

#alg_bench = pd.unique(trades['productname'])
## Get a list of all files in the 'algoritmos' folder
algo_files = [f for f in os.listdir(algoritmos_path) if f.endswith('.csv')]
algo_files = [os.path.join(algoritmos_path, name) for name in algo_files]
#if algo_files:
#    algs_benchmark = [os.path.join(algoritmos_path, name + ".csv") for name in alg_bench]
#else:
#    print(f"No CSV files found in '{algoritmos_path}'")

In [20]:
len(algo_files[:400:5])

80

In [21]:
horas = set()
fecha_max = datetime(2019,12,31,0,0,0)
fecha_min = datetime(2025,12,31,0,0,0)
for algo in algo_files[:3000]:
    df = pd.read_csv(algo)
    df['plot_date'] = pd.to_datetime(df['datetime'], format = "%Y-%m-%d %H:%M:%S")
    horas.update(pd.unique(df['plot_date']))
    fecha_max_algo = df['plot_date'].max()
    fecha_min_algo = df['plot_date'].min()
    if fecha_max_algo > fecha_max:
        fecha_max = fecha_max_algo
    if fecha_min_algo< fecha_min:
        fecha_min = fecha_min_algo
horas = sorted(list(horas))

In [22]:
df_algoritmos = pd.DataFrame()
df_algoritmos['Date'] = horas

for algo in algo_files[:3000]:
    name = algo.split('/')[-1].replace('.csv','')
    df = pd.read_csv(algo)
    df['plot_date'] = pd.to_datetime(df['datetime'], format = "%Y-%m-%d %H:%M:%S")
    df_merge = df[['plot_date','open']]
    df_merge.rename(columns = {'open':name}, inplace=True)
    df_algoritmos = df_algoritmos.merge(df_merge[['plot_date',name]], how='left', left_on='Date', right_on='plot_date')
    df_algoritmos.drop(columns = 'plot_date', inplace = True)

#tenemos aquí todos los precios de apertura de todos los algoritmos usados en la cartera a lo largo de sus vidas.
df_algoritmos.fillna(0, inplace=True)

,Date,DTgLV,nMhR4,4lUfK,J8oVx,qdIrZ,imHBP,6eXdG,Re99P,4sg4R,...,1YQc4,EqAOG,zf6bf,2uvog,uKlpA,fUr64,n5iI8,sak7E,fS6k5,ulNJ2
0,2020-06-01 01:00:00,0.00,130.18,0.0,0.0,0.00,0.0,133.32,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
1,2020-06-01 05:00:00,0.00,130.15,0.0,0.0,0.00,0.0,133.61,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
2,2020-06-01 09:00:00,0.00,130.27,0.0,0.0,0.00,0.0,133.74,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
3,2020-06-01 13:00:00,0.00,129.89,0.0,0.0,0.00,0.0,133.08,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
4,2020-06-01 17:00:00,0.00,129.12,0.0,0.0,99.94,0.0,133.52,0.0,0,...,99.02,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7141,2024-12-30 06:00:00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
7142,2024-12-30 10:00:00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,95.73,0.0
7143,2024-12-30 14:00:00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,96.23,0.0
7144,2024-12-30 18:00:00,92.48,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,96.22,0.0


In [33]:
df_algoritmos

,Date,DTgLV,nMhR4,4lUfK,J8oVx,qdIrZ,imHBP,6eXdG,Re99P,4sg4R,...,1YQc4,EqAOG,zf6bf,2uvog,uKlpA,fUr64,n5iI8,sak7E,fS6k5,ulNJ2
0,2020-06-01 01:00:00,0.00,130.18,0.0,0.0,0.00,0.0,133.32,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
1,2020-06-01 05:00:00,0.00,130.15,0.0,0.0,0.00,0.0,133.61,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
2,2020-06-01 09:00:00,0.00,130.27,0.0,0.0,0.00,0.0,133.74,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
3,2020-06-01 13:00:00,0.00,129.89,0.0,0.0,0.00,0.0,133.08,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
4,2020-06-01 17:00:00,0.00,129.12,0.0,0.0,99.94,0.0,133.52,0.0,0,...,99.02,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7141,2024-12-30 06:00:00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,0.00,0.0
7142,2024-12-30 10:00:00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,95.73,0.0
7143,2024-12-30 14:00:00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,96.23,0.0
7144,2024-12-30 18:00:00,92.48,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0,...,0.00,0.0,0.0,0.0,0.0,0.0,0,0.0,96.22,0.0


In [34]:
df = df_algoritmos.copy()

In [50]:
# df is your input DataFrame
timestamp = df.iloc[:, 0].values        # shape: (7146,)
series_data = df.iloc[:, 1:].astype(float).to_numpy()     # shape: (7146, 3000)
column_names = df.columns[1:]#.astype(float).to_numpy()           # store these for later


In [51]:
def create_dynamic_windowed_dataset(data, window, n_future, activity_threshold=0.0):
    """
    data: shape (T, N_features)
    window: number of timesteps for X
    n_future: number of timesteps for Y
    """
    X, Y = [], []
    T, N = data.shape

    for i in range(T - window - n_future + 1):
        x_window = data[i : i + window]
        y_window = data[i + window : i + window + n_future]

        # activity filter: skip windows that are too empty / zero
        activity = np.sum(np.abs(x_window))

        if activity <= activity_threshold:
            continue

        X.append(x_window)
        Y.append(y_window)

    return np.array(X), np.array(Y)


In [52]:
window = 30
n_future = 20
activity_threshold = 1e-6  # Adjust depending on sparsity

X, Y = create_dynamic_windowed_dataset(series_data, window, n_future, activity_threshold)

print("X shape:", X.shape)  # Expected: (samples, 30, 3000)
print("Y shape:", Y.shape)  # Expected: (samples, 20, 3000)


X shape: (7097, 30, 3000)
Y shape: (7097, 20, 3000)


In [53]:
N_features = series_data.shape[1]
output_dim = n_future * N_features    # flatten target

model = Sequential()
model.add(Masking(mask_value=0.0, input_shape=(window, N_features)))
model.add(LSTM(128))
model.add(Dense(output_dim))

model.compile(optimizer="adam", loss="mse")


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [55]:
Y_flat = Y.reshape(len(Y), -1)
model.fit(X, Y_flat, epochs=20, batch_size=32, validation_split=0.1)


Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 72ms/step - loss: 1638.4139 - val_loss: 1510.3978
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 17s 86ms/step - loss: 1360.5707 - val_loss: 1437.0889
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - loss: 1246.0841 - val_loss: 1420.6902
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 14s 68ms/step - loss: 1187.2303 - val_loss: 1404.8672
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 14s 69ms/step - loss: 1152.1079 - val_loss: 1398.6492
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 14s 68ms/step - loss: 1128.6857 - val_loss: 1395.7450
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 78ms/step - loss: 1111.1378 - val_loss: 1384.2843
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 78ms/step - loss: 1097.9626 - val_loss: 1378.0400
Epoch 9/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 78ms/step - loss: 1085.7266 - val_loss: 1369.8657
Epoch 10/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - loss: 1072.9772 - val_loss: 1362.2944
Epoch 11/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 81ms/step - lo

In [56]:

model.save("my_predictor.keras")


In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
model = load_model("your_model.h5")

new_series = np.array(your_last_30_values)

new_series = new_series.reshape(1, 30, 1)
prediction = model.predict(new_series)
print("Predicted next 20 values:")
print(prediction)

In [ ]:
df_algoritmos

In [ ]:
#necesitamos una función que normalice funciones con una gaussiana a un dominio común del intervalo [0,1]
def gaussian_kernel_smoother(x, y, x_eval, bandwidth):
    x = np.asarray(x).flatten()
    y = np.asarray(y).flatten()
    x_eval = np.asarray(x_eval).flatten()

    # Compute squared distance matrix
    diff = x_eval[:, None] - x[None, :]

    # Gaussian kernel weights
    weights = np.exp(-0.5 * (diff / bandwidth)**2)

    # Normalize weights
    weights /= np.sum(weights, axis=1, keepdims=True)

    # Smoothed values
    y_smooth = weights @ y
    return y_smooth


In [ ]:
# X_train shape: (samples, window, 1)
# y_train shape: (samples, n_future)

model = models.Sequential()
model.add(layers.LSTM(64, return_sequences=False, input_shape=(window, 1)))
model.add(layers.Dense(n_future))

model.compile(optimizer='adam', loss='mse')

model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_val, y_val))

pred = model.predict(X_val[-1].reshape(1, window, 1))


In [ ]:
# trades_coarse_functions = []
# for index, row in df_completo.iterrows():
#     precios = [df_algoritmos.loc[(df_algoritmos['Date']>=row['prox_open_hour']) & (df_algoritmos['Date']<=row['prox_close_hour']),row['productname']].values]
#     trades_coarse_functions.append([index, np.linspace(0,1,row['n_filas']),np.array(precios)/np.array(precios).mean()])

# x_common = np.linspace(0,1,200)
# smooth_functions = []
# bandwidth = 0.04
# for index_i,x_i,y_i in trades_coarse_functions:
#     smooth_functions.append(gaussian_kernel_smoother(x_i,y_i, x_common,bandwidth))

In [ ]:
# trades_history_coarse_functions = []
# for index, row in df_completo.iterrows():
#     precios_previos = df_algoritmos.loc[(df_algoritmos['Date']<=row['prox_open_hour']) & (df_algoritmos[row['productname']]!=0),row['productname']].values
#     trades_history_coarse_functions.append([index, np.linspace(0,1,len(precios_previos[:35])),np.array(precios_previos[:35])/np.array(precios_previos[:35]).mean()])

# history_smooth_functions = []
# x_common = np.linspace(0,1,200)
# andwidth = 0.04
# for index_i,x_i,y_i in trades_history_coarse_functions:
#     history_smooth_functions.append(gaussian_kernel_smoother(x_i,y_i, x_common,bandwidth))
